# Representative Statements Workbench

The purpose of this notebook is to test the Agora representative statement selection and compare it to the Polis output for the same clusters. 

In [1]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", None)

REPO_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "pyproject.toml").exists() and (parent / "reddwarf").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import reddwarf
imported_repo_root = Path(reddwarf.__file__).resolve().parents[1]
if imported_repo_root != REPO_ROOT:
    raise RuntimeError(
        f"Notebook imported reddwarf from {imported_repo_root}, expected {REPO_ROOT}. Restart the kernel and rerun from the repo clone."
    )

from reddwarf.data_loader import Loader
from reddwarf.utils.matrix import generate_raw_matrix
from reddwarf.utils.polismath import extract_data_from_polismath
from reddwarf.utils.stats import (
    calculate_comment_statistics_dataframes,
    rank_representative_statements,
    select_representative_statements,
)
from reddwarf.utils.statements import process_statements


c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'exclude' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'exclude' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


## Settings

- `DATA_SOURCE = "fixture"` uses local test fixtures
- `DATA_SOURCE = "polis_id"` fetches a live report by Polis report id
- This notebook always reuses platform `math_data["group-clusters"]` for both Polis and Agora representative selection so the cluster source stays fixed.


In [2]:
DATA_SOURCE = "fixture"  # fixture | polis_id
FIXTURE_DIR = "../../tests/fixtures/below-100-ptpts"
POLIS_ID = ""

RANDOM_STATE = 42
FDR_RATE = 0.10
DIVISIVE_N_RESAMPLES = 399
DIVISIVE_RANDOM_STATE = 42
STRONG_EFFECT_MIN = 1.0
STRONG_SMALL_GROUP_CUTOFF = 5
STRONG_LARGE_GROUP_PARTICIPATION_MIN = 0.8
STRONG_P_MAX = 0.05
TOP_N = 10


In [3]:
if DATA_SOURCE == "fixture":
    loader = Loader(filepaths=[
        f"{FIXTURE_DIR}/votes.json",
        f"{FIXTURE_DIR}/comments.json",
        f"{FIXTURE_DIR}/conversation.json",
        f"{FIXTURE_DIR}/math-pca2.json",
    ])
    dataset_label = FIXTURE_DIR
elif DATA_SOURCE == "polis_id":
    loader = Loader(polis_id=POLIS_ID)
    dataset_label = POLIS_ID
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

_, _, mod_out_statement_ids, meta_statement_ids = process_statements(loader.comments_data)

stmt_text = {}
for comment in loader.comments_data:
    sid = comment.get("tid", comment.get("statement_id"))
    stmt_text[int(sid)] = comment.get("txt", "")

if not getattr(loader, "math_data", None) or "group-clusters" not in loader.math_data:
    raise ValueError("This notebook requires platform math_data with group-clusters.")

print(f"Dataset: {dataset_label}")
print(f"Votes: {len(loader.votes_data)}")
print(f"Statements: {len(loader.comments_data)} total")
print(f"Moderated out / meta: {len(mod_out_statement_ids)}")
print("Cluster source: platform_group_clusters")


Dataset: ../../tests/fixtures/below-100-ptpts
Votes: 504
Statements: 44 total
Moderated out / meta: 2
Cluster source: platform_group_clusters


## Building Shared Context for Both Pipelines

This notebook reuses the platform (from Polis) participant-to-group assignments from `math_data["group-clusters"]` and does not rerun clustering (for the sake of comparing the pipelines on identical clusters)


In [4]:
def get_platform_cluster_context(loader, valid_participant_ids):
    platform_ids, platform_labels = extract_data_from_polismath(loader.math_data)
    valid_ids = set(valid_participant_ids)
    platform_pairs = [
        (int(pid), int(label))
        for pid, label in zip(platform_ids, platform_labels)
        if pid in valid_ids
    ]
    participant_ids_to_cluster = [pid for pid, _ in platform_pairs]
    cluster_labels = np.asarray([label for _, label in platform_pairs], dtype=int)
    return participant_ids_to_cluster, cluster_labels

raw_vote_matrix = generate_raw_matrix(votes=loader.votes_data)
participant_ids_to_cluster, cluster_labels = get_platform_cluster_context(loader, raw_vote_matrix.index)
clustered_vote_matrix = raw_vote_matrix.loc[participant_ids_to_cluster, :]

grouped_stats_df, _ = calculate_comment_statistics_dataframes(
    vote_matrix=clustered_vote_matrix,
    cluster_labels=cluster_labels,
    consensus_mode="standard",
)

ranked_repness = rank_representative_statements(
    grouped_stats_df=grouped_stats_df,
    vote_matrix=clustered_vote_matrix,
    cluster_labels=cluster_labels,
    mod_out_statement_ids=mod_out_statement_ids,
    fdr_rate=FDR_RATE,
    divisive_n_resamples=DIVISIVE_N_RESAMPLES,
    divisive_random_state=DIVISIVE_RANDOM_STATE,
    strong_effect_min=STRONG_EFFECT_MIN,
    strong_small_group_cutoff=STRONG_SMALL_GROUP_CUTOFF,
    strong_large_group_participation_min=STRONG_LARGE_GROUP_PARTICIPATION_MIN,
    strong_p_max=STRONG_P_MAX,
)

participants_df = pd.DataFrame(index=raw_vote_matrix.index)
participants_df["to_cluster"] = participants_df.index.isin(participant_ids_to_cluster)
participants_df["cluster_id"] = pd.Series(cluster_labels, index=participant_ids_to_cluster, dtype="Int64")

agora_result = SimpleNamespace(
    raw_vote_matrix=raw_vote_matrix,
    group_comment_stats=grouped_stats_df,
    ranked_repness=ranked_repness,
    participants_df=participants_df,
)

print(f"Agora groups: {sorted(agora_result.ranked_repness.keys())}")
print(f"Agora clustered participants: {int(agora_result.participants_df['to_cluster'].sum())}")


Agora groups: [0, 1, 2]
Agora clustered participants: 23


In [5]:
polis_repness_same_groups = select_representative_statements(
    grouped_stats_df=agora_result.group_comment_stats,
    mod_out_statement_ids=mod_out_statement_ids,
    pick_max=5,
    confidence=0.9,
)

print(f"Polis groups on same cluster source: {sorted(polis_repness_same_groups, key=int)}")


Polis groups on same cluster source: [0, 1, 2]


## View Representative Statements for Both Pipelines

In [6]:
def truncate(text, max_len=100):
    text = text or ""
    return text if len(text) <= max_len else text[:max_len] + "..."

def vote_breakdown(na, nd, ns):
    na = int(na)
    nd = int(nd)
    ns = int(ns)
    npass = max(ns - na - nd, 0)
    votes = f"{na}/{nd}/{npass}/{ns}"
    if ns == 0:
        pct = "0.0%/0.0%/0.0%"
    else:
        pct = f"{100*na/ns:.1f}%/{100*nd/ns:.1f}%/{100*npass/ns:.1f}%"
    return votes, pct

def polis_group_df(group_id):
    rows = []
    for rank, row in enumerate(polis_repness_same_groups.get(group_id, []), start=1):
        statement_id = int(row["tid"])
        grouped_row = agora_result.group_comment_stats.loc[(group_id, statement_id)]
        in_votes, in_pct = vote_breakdown(grouped_row["na"], grouped_row["nd"], grouped_row["ns"])
        rows.append({
            "rank": rank,
            "statement_id": statement_id,
            "repful_for": row.get("repful-for"),
            "best_agree": bool(row.get("best-agree", False)),
            "n_success": int(row["n-success"]),
            "n_trials": int(row["n-trials"]),
            "p_success": float(row["p-success"]),
            "p_test": float(row["p-test"]),
            "repness": float(row["repness"]) if row.get("repness") is not None else None,
            "repness_test": float(row["repness-test"]) if row.get("repness-test") is not None else None,
            "In Votes": in_votes,
            "In %": in_pct,
            "text": truncate(stmt_text.get(statement_id, "?")),
        })
    return pd.DataFrame(rows)

def agora_group_df_detailed(group_id, top_n=None):
    rows = []
    statements = agora_result.ranked_repness[group_id]
    if top_n is not None:
        statements = statements[:top_n]
    for statement in statements:
        npass = int(statement.ns - statement.na - statement.nd)
        npass_out = int(statement.ns_out - statement.na_out - statement.nd_out)
        in_votes, in_pct = vote_breakdown(statement.na, statement.nd, statement.ns)
        out_votes, out_pct = vote_breakdown(statement.na_out, statement.nd_out, statement.ns_out)
        rows.append({
            "rank": int(statement.rank),
            "st_id": int(statement.statement_id),
            "repful_for": statement.repful_for,
            "selected": bool(statement.selected),
            "strength": statement.signal_strength,
            "effect_size": float(statement.effect_size),
            "p_value": float(statement.p_value),
            "adjusted_p_value": float(statement.adjusted_p_value),
            "agree_effect": float(statement.agree_effect),
            "disagree_effect": float(statement.disagree_effect),
            "divisive_effect": float(statement.divisive_effect),
            "na": int(statement.na),
            "nd": int(statement.nd),
            "npass": npass,
            "ns": int(statement.ns),
            "In Votes": in_votes,
            "In %": in_pct,
            "na_out": int(statement.na_out),
            "nd_out": int(statement.nd_out),
            "npass_out": npass_out,
            "ns_out": int(statement.ns_out),
            "Out Votes": out_votes,
            "Out %": out_pct,
            "divisiveness": float(statement.divisiveness),
            "text": truncate(stmt_text.get(statement.statement_id, "?")),
        })
    return pd.DataFrame(rows)

print(f"Dataset: {dataset_label}")
for gid in sorted(agora_result.ranked_repness):
    print(f"\n=== Group {gid} ===\n")
    print("Polis representative statements (same cluster source):")
    display(polis_group_df(gid))
    print("\nAgora representative statements:")
    display(agora_group_df_detailed(gid, top_n=TOP_N))


Dataset: ../../tests/fixtures/below-100-ptpts

=== Group 0 ===

Polis representative statements (same cluster source):


,rank,statement_id,repful_for,best_agree,n_success,n_trials,p_success,p_test,repness,repness_test,In Votes,In %,text
0,1,13,agree,True,6,6,0.875000,2.645751,2.625000,2.699862,6/0/0/6,100.0%/0.0%/0.0%,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
1,2,4,agree,False,6,6,0.875000,2.645751,2.100000,2.393172,6/0/0/6,100.0%/0.0%/0.0%,There's no way to review previous statements (mine and others). This makes it hard to circle back to...
2,3,2,agree,False,4,5,0.714286,1.632993,3.928571,2.472280,4/1/0/5,80.0%/20.0%/0.0%,"There's no way to differentiate between ""disagree"" and ""pass"" votes. This is misleading as the 2 pos..."
3,4,19,agree,False,7,8,0.800000,2.333333,2.400000,2.385421,7/1/0/8,87.5%/12.5%/0.0%,Polis feels too abstract and alienating since there is no context or background to justify a positio...
4,5,24,agree,False,5,5,0.857143,2.449490,2.057143,2.248967,5/0/0/5,100.0%/0.0%/0.0%,Polis needs to support translations for statements



Agora representative statements:


,rank,st_id,repful_for,selected,strength,effect_size,p_value,adjusted_p_value,agree_effect,disagree_effect,divisive_effect,na,nd,npass,ns,In Votes,In %,na_out,nd_out,npass_out,ns_out,Out Votes,Out %,divisiveness,text
0,1,21,agree,False,normal,2.479339,0.029391,0.108746,2.479339,0.000000,0.000000,5,0,1,6,5/0/1/6,83.3%/0.0%/16.7%,1,4,1,6,1/4/1/6,16.7%/66.7%/16.7%,0.000000,"I want to be able to share more complex, long-form ideas instead of 140 character statements"
1,2,19,agree,True,normal,1.619835,0.009815,0.066165,1.619835,0.024793,0.066116,7,1,0,8,7/1/0/8,87.5%/12.5%/0.0%,3,4,3,10,3/4/3/10,30.0%/40.0%/30.0%,0.181818,Polis feels too abstract and alienating since there is no context or background to justify a positio...
2,3,2,agree,True,normal,1.586777,0.013425,0.066165,1.586777,0.033058,0.198347,4,1,0,5,4/1/0/5,80.0%/20.0%/0.0%,1,3,5,9,1/3/5/9,11.1%/33.3%/55.6%,0.181818,"There's no way to differentiate between ""disagree"" and ""pass"" votes. This is misleading as the 2 pos..."
3,4,9,agree,False,normal,1.239669,0.039673,0.133445,1.239669,0.024793,0.099174,5,1,1,7,5/1/1/7,71.4%/14.3%/14.3%,2,4,4,10,2/4/4/10,20.0%/40.0%/40.0%,0.181818,"Due to the 140 character limit, users tend to give up on communicating complex ideas."
4,5,13,agree,True,normal,1.190083,0.004075,0.066165,1.190083,0.000000,0.000000,6,0,0,6,6/0/0/6,100.0%/0.0%/0.0%,3,4,3,10,3/4/3/10,30.0%/40.0%/30.0%,0.000000,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
5,6,14,agree,True,normal,1.190083,0.019981,0.082145,1.190083,0.000000,0.000000,6,0,1,7,6/0/1/7,85.7%/0.0%/14.3%,3,2,4,9,3/2/4/9,33.3%/22.2%/44.4%,0.000000,"Polis should have a ""remind me later"" feature so busy people can go through all statements as per th..."
6,7,4,agree,True,normal,0.892562,0.008151,0.066165,0.892562,0.000000,0.000000,6,0,0,6,6/0/0/6,100.0%/0.0%/0.0%,4,2,4,10,4/2/4/10,40.0%/20.0%/40.0%,0.000000,There's no way to review previous statements (mine and others). This makes it hard to circle back to...
7,8,34,agree,False,normal,0.892562,0.171391,0.294731,0.892562,0.000000,0.000000,3,0,1,4,3/0/1/4,75.0%/0.0%/25.0%,1,1,1,3,1/1/1/3,33.3%/33.3%/33.3%,0.000000,It asks for my email once I’ve got through all the statements but I’m not sure where that data goes ...
8,9,27,agree,False,normal,0.793388,0.051235,0.145823,0.793388,0.000000,0.000000,4,0,1,5,4/0/1/5,80.0%/0.0%/20.0%,2,1,4,7,2/1/4/7,28.6%/14.3%/57.1%,0.000000,I wish statements weren't limited to only 140 characters and there was more clarity about how statem...
9,10,28,agree,True,normal,0.714050,0.008151,0.066165,0.714050,0.000000,0.000000,6,0,0,6,6/0/0/6,100.0%/0.0%/0.0%,5,0,0,5,5/0/0/5,100.0%/0.0%/0.0%,0.000000,Questions that have been 'passed' should be available to answer later since that is an implication o...



=== Group 1 ===

Polis representative statements (same cluster source):


...

,rank,st_id,repful_for,selected,strength,effect_size,p_value,adjusted_p_value,agree_effect,disagree_effect,divisive_effect,na,nd,npass,ns,In Votes,In %,na_out,nd_out,npass_out,ns_out,Out Votes,Out %,divisiveness,text
0,1,4,disagree,True,strong,8.888889,0.003990,0.053868,0.246914,8.888889,4.444444,1,2,0,3,1/2/0/3,33.3%/66.7%/0.0%,9,0,4,13,9/0/4/13,69.2%/0.0%/30.8%,0.666667,There's no way to review previous statements (mine and others). This makes it hard to circle back to...
1,2,10,agree,False,normal,8.888889,0.022121,0.122851,8.888889,0.740741,4.444444,2,1,0,3,2/1/0/3,66.7%/33.3%/0.0%,1,3,8,12,1/3/8/12,8.3%/25.0%/66.7%,0.666667,Comments should follow a narrative structure to reduce participant fatigue.
2,3,16,disagree,True,strong,8.888889,0.002896,0.053868,0.246914,8.888889,4.444444,1,2,0,3,1/2/0/3,33.3%/66.7%/0.0%,9,0,5,14,9/0/5/14,64.3%/0.0%/35.7%,0.666667,It sucks that there's no way to undo a vote
3,4,17,agree,False,normal,5.000000,0.022750,0.122851,5.000000,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,4,2,7,13,4/2/7/13,30.8%/15.4%/53.8%,0.000000,I have no way to communicate that 1 specific statement is more important to me than others
4,5,20,agree,False,normal,4.000000,0.022750,0.122851,4.000000,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,5,5,3,13,5/5/3/13,38.5%/38.5%/23.1%,0.000000,It's hard to communicate technical and quantitative points-of-view in Polis
5,6,22,disagree,False,normal,3.333333,0.035220,0.122851,0.000000,3.333333,0.000000,0,3,0,3,0/3/0/3,0.0%/100.0%/0.0%,4,6,3,13,4/6/3/13,30.8%/46.2%/23.1%,0.000000,Polis relies on the participant to put themselves in the other person's shoes before voting
6,7,13,agree,False,normal,3.333333,0.035220,0.122851,3.333333,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,6,4,3,13,6/4/3/13,46.2%/30.8%/23.1%,0.000000,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
7,8,1,disagree,False,normal,2.962963,0.115993,0.225457,0.000000,2.962963,0.000000,0,2,1,3,0/2/1/3,0.0%/66.7%/33.3%,4,3,5,12,4/3/5/12,33.3%/25.0%/41.7%,0.000000,The prominent display of divisive issues is concerning as politicians like to find wedge issues and ...
8,9,0,disagree,False,normal,2.500000,0.045500,0.122851,0.000000,2.500000,0.000000,0,3,0,3,0/3/0/3,0.0%/100.0%/0.0%,0,8,3,11,0/8/3/11,0.0%/72.7%/27.3%,0.000000,Polis is perfect just the way it is. No changes are necessary.
9,10,15,disagree,False,normal,2.222222,0.129714,0.225457,0.555556,2.222222,4.444444,1,1,1,3,1/1/1/3,33.3%/33.3%/33.3%,4,1,8,13,4/1/8/13,30.8%/7.7%/61.5%,0.666667,The inability to search for statements as a moderator limits me



=== Group 2 ===

Polis representative statements (same cluster source):


,rank,statement_id,repful_for,best_agree,n_success,n_trials,p_success,p_test,repness,repness_test,In Votes,In %,text
0,1,32,agree,True,3,3,0.8,2.0,1.6,1.247219,3/0/0/3,100.0%/0.0%/0.0%,There should be a QR code for sharing poll



Agora representative statements:


,rank,st_id,repful_for,selected,strength,effect_size,p_value,adjusted_p_value,agree_effect,disagree_effect,divisive_effect,na,nd,npass,ns,In Votes,In %,na_out,nd_out,npass_out,ns_out,Out Votes,Out %,divisiveness,text
0,1,13,disagree,False,normal,2.765432,0.018881,0.246929,0.000000,2.765432,0.000000,0,4,3,7,0/4/3/7,0.0%/57.1%/42.9%,9,0,0,9,9/0/0/9,100.0%/0.0%/0.0%,0.000000,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
1,2,9,disagree,False,normal,2.765432,0.048014,0.246929,0.000000,2.765432,0.000000,0,4,3,7,0/4/3/7,0.0%/57.1%/42.9%,7,1,2,10,7/1/2/10,70.0%/10.0%/20.0%,0.000000,"Due to the 140 character limit, users tend to give up on communicating complex ideas."
2,3,21,disagree,False,normal,2.765432,0.030754,0.246929,0.034568,2.765432,0.345679,1,4,1,6,1/4/1/6,16.7%/66.7%/16.7%,5,0,1,6,5/0/1/6,83.3%/0.0%/16.7%,0.222222,"I want to be able to share more complex, long-form ideas instead of 140 character statements"
3,4,31,agree,False,normal,1.555556,0.179712,0.497665,1.555556,0.000000,0.000000,3,0,1,4,3/0/1/4,75.0%/0.0%/25.0%,1,0,1,2,1/0/1/2,50.0%/0.0%/50.0%,0.000000,Nuance in Polis is created by entering multiple statements.
4,5,32,agree,False,normal,1.555556,0.045500,0.246929,1.555556,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,1,0,1,2,1/0/1/2,50.0%/0.0%/50.0%,0.000000,There should be a QR code for sharing poll
5,6,14,divisive,False,normal,1.382716,0.135000,0.441818,0.098765,0.691358,1.382716,2,2,2,6,2/2/2/6,33.3%/33.3%/33.3%,7,0,3,10,7/0/3/10,70.0%/0.0%/30.0%,0.444444,"Polis should have a ""remind me later"" feature so busy people can go through all statements as per th..."
6,7,1,divisive,False,normal,1.382716,0.080000,0.320000,1.555556,0.230453,1.382716,3,2,2,7,3/2/2/7,42.9%/28.6%/28.6%,1,3,4,8,1/3/4/8,12.5%/37.5%/50.0%,0.444444,The prominent display of divisive issues is concerning as politicians like to find wedge issues and ...
7,8,6,divisive,False,normal,1.382716,0.110000,0.396000,0.259259,0.691358,1.382716,3,2,3,8,3/2/3/8,37.5%/25.0%/37.5%,6,0,3,9,6/0/3/9,66.7%/0.0%/33.3%,0.444444,"It's unclear if Polis' algorithms are resilient to language nuances (sarcasm, dialects, abbreviation..."
8,9,22,divisive,False,normal,1.382716,0.150000,0.450000,1.555556,0.098765,1.382716,3,2,2,7,3/2/2/7,42.9%/28.6%/28.6%,1,7,1,9,1/7/1/9,11.1%/77.8%/11.1%,0.444444,Polis relies on the participant to put themselves in the other person's shoes before voting
9,10,25,disagree,False,normal,1.382716,0.229142,0.557415,0.034568,1.382716,0.172840,1,4,2,7,1/4/2/7,14.3%/57.1%/28.6%,5,2,1,8,5/2/1/8,62.5%/25.0%/12.5%,0.222222,I want to be able to make a statement referring to an existing statement so I can provide more conte...


## Summary

Comparison of Polis top 5 ids, Agora top-ranked ids and Agora `selected=True` ids per group.


In [7]:
summary_rows = []
for gid in sorted(agora_result.ranked_repness):
    polis_top_ids = [int(row["tid"]) for row in polis_repness_same_groups.get(gid, [])]
    agora_top_ids = [statement.statement_id for statement in agora_result.ranked_repness[gid][:TOP_N]]
    agora_selected_ids = [statement.statement_id for statement in agora_result.ranked_repness[gid] if statement.selected]
    summary_rows.append({
        "group_id": gid,
        "polis_top_ids": polis_top_ids,
        "agora_top_ids": agora_top_ids,
        "agora_selected_ids": agora_selected_ids,
        "n_agora_selected": len(agora_selected_ids),
    })

display(pd.DataFrame(summary_rows))


,group_id,polis_top_ids,agora_top_ids,agora_selected_ids,n_agora_selected
0,0,"[13, 4, 2, 19, 24]","[21, 19, 2, 9, 13, 14, 4, 34, 27, 28]","[19, 2, 13, 14, 4, 28, 18, 24, 0]",9
1,1,"[17, 20, 13, 27, 22]","[4, 10, 16, 17, 20, 22, 13, 1, 0, 15]","[4, 16]",2
2,2,[32],"[13, 9, 21, 31, 32, 14, 1, 6, 22, 25]",[],0
